# Results for model: mistralai/mistral-7b-instruct-v0.2

In [ ]:
import polars as pl
import xgboost as xgb

# Load data using Polars
data = pl.read_parquet("./jane_street_data/train.parquet")

# Feature Engineering: Calculate a global TOP_QUANTILE of 'feature_00' relative to 'responder_6'
# across rolling batches of 'date_id'
window_size = 150  # You may adjust the window size as required
window = pl. Window.rolling(data, 'date_id', window_size=window_size)

top_quantile_series = data.select(
    pl.expr(
        f"""
        quantile({pl.col('feature_00')}, 0.15) over ({window.order_by('date_id')}) as top_quanta
        """
    )
).select(pl.col('date_id'), pl.col('feature_00'), pl.col('responder_6'), 'top_quanta')

# Merge top quantile series with original dataframe
data_with_topquanta = top_quantile_series.join(data, on={'date_id': 'date_id'})

# Rename columns for XGBoost Regressor
X = data_with_topquanta.select([pl.col('feature_00'), 'top_quanta']).rename(
    columns={'feature_00': 'field_0', 'top_quanta': 'target'}
)
y = data_with_topquanta.select('responder_6')

# Train an XGBoost Regressor on the target 'responder_6'
xgb_model = xgb.XGBRegressor()
xgb_model.fit(X.to_pandas().values, y.to_pandas().values)

# Optional: Save the model
import joblib
joblib.dump(xgb_model, 'xgboost_model.pkl')